In [691]:
from openpyxl import load_workbook
from collections import namedtuple
import pandas as pd
import re

In [ ]:
"""
from openpyxl import load_workbook
wb = load_workbook(filename = 'empty_book.xlsx')
sheet_ranges = wb['range names']
print(sheet_ranges['D18'].value)
"""

In [16]:
wb = load_workbook(filename="001 Laskentapohja, 2024.xlsx", data_only=True)

In [4]:
wb.sheetnames

['Tuntihinnoittelut',
 'Materiaalihinnoittelut',
 'Purkutyöt',
 'Väliseinät sisällä',
 'Maalaukset',
 'Alakatot',
 'Laatoitukset',
 'Lattiat',
 'Oviasennukset ja listoitukset',
 'Erillishankinnat',
 'Eritelysivu',
 'Koontisivu',
 'Massalista, rak. tarvikkeet ',
 'Päivitykset pohjaan']

In [152]:
print(wb['Tuntihinnoittelut'].calculate_dimension())
print(wb['Tuntihinnoittelut'].max_row,',', wb['Tuntihinnoittelut'].max_column, sep='')

A1:C39
39,3


In [39]:
wb['Materiaalihinnoittelut'].max_row, wb['Materiaalihinnoittelut'].max_column

(327, 7)

In [145]:
for sn in wb.sheetnames:
    print(f'{sn}: {wb[sn].max_row, wb[sn].max_column}')

Tuntihinnoittelut: (39, 3)
Materiaalihinnoittelut: (327, 7)
Purkutyöt: (71, 8)
Väliseinät sisällä: (349, 28)
Maalaukset: (736, 18)
Alakatot: (240, 13)
Laatoitukset: (330, 18)
Lattiat: (279, 14)
Oviasennukset ja listoitukset: (272, 13)
Erillishankinnat: (79, 9)
Eritelysivu: (131, 7)
Koontisivu: (41, 4)
Massalista, rak. tarvikkeet : (365, 16)
Päivitykset pohjaan: (3, 7)


In [597]:
# tuntihinnoittelut
lines = namedtuple('item', ['pos', 'kuvaus', 'tunnit'])
rows = [lines(*row) for row in wb['Tuntihinnoittelut'].iter_rows(values_only=True) if row.count(None) < 2]
tuntihinnoittelut_cols = pd.DataFrame(rows).iloc[:, 1:]
tuntihinnoittelut_cols

,kuvaus,tunnit
0,Tuntihinta,42
1,Kuvaus,Tunnit
2,Työmaan perustus + työmaataulu ja ilmoitukset,16
3,Suojaukset,16
4,Vaneriseinän rakennus,0
5,wc alakaton purku ja uudelleen rakennus,30
6,Haalaukset,40
7,IV-tulppaukset,0
8,Kaihdinten kunnon tarkastus ja huolto,0
9,Apurungon rakennus,16


In [241]:
# materiaalihinnoittelu

line = namedtuple('item', ['POS', 'Tuote', 'Hinta', 'Kuvaus', 'kpl', 'Yht', 'ignore'])

a = [line(*row) for row in wb['Materiaalihinnoittelut'].iter_rows(values_only=True)]
b = [row for row in a if isinstance(row.Yht, (int, float)) and not row.Yht == 0 and not row.Tuote == 'Yhteensä']

In [580]:
# erittelysivu

line1 = namedtuple('item', ['selite', 'materiaalit', 'työt'])
line2 = namedtuple('item', ['Erillishankinnat', 'Hinta', 
                            'Toimittaja_urakoitsija', 'ignore1', 
                            'Toimitus', 'ignore2', 'ignore3'])

erittelyt_cols_left = []
erittelyt_cols_right = []
erillishankinnat_cols = []
kokonaishinnat = []

temp = []

for i, row in enumerate(wb['Eritelysivu'].iter_rows(values_only=True)):
    if i < 77:
        erittelyt_cols_left.append(line1(*row[:3]))
        erittelyt_cols_right.append(line1(*row[4:7]))
    if 77 <= i < 97:
        erittelyt_cols_left.append(line1(*row[:3]))
    if 97 <= i < 114:
        erillishankinnat_cols.append(line2(*row))
    if i > 114:
        if row.count(None) < 5:
            if len(temp) == 0:
                ntuple_names = list(row)
                ntuple_names = [f'id_{i}' if x is None else x for i, x in enumerate(ntuple_names)]
                ntuple_names = [x.replace('-', '_') for x in ntuple_names]
                ntuple = namedtuple('item', ntuple_names)
                temp.append(ntuple)
                continue
            if len(temp) != 0:
                out = temp.pop()
                out = out(*row)
                kokonaishinnat.append(out)


df_erillishankinnat = pd.DataFrame(erillishankinnat_cols).iloc[:, [0, 1,4]]

df_left = pd.DataFrame(erittelyt_cols_left)
df_right = pd.DataFrame(erittelyt_cols_right)
df_right.columns = df_left.columns
df_erittelyt = pd.concat([df_left, df_right])

# TODO: tarvitaanko kokonaishinnat jos on jo vastaavat erittelysivulla?
[a._asdict() for a in kokonaishinnat]


[{'Omakustannehinta': None,
  'Työt': 45487.3,
  'Materiaalit': 66004.51108,
  'id_3': None,
  'Erillishankinnat': 49294,
  'id_5': None,
  'id_6': None},
 {'Kate_osuus': None,
  'Työ': 6663.405882352942,
  'Materiaalit': 11519.750321307189,
  'id_3': None,
  'Erillishankinnat': 8698.94117647059,
  'id_5': None,
  'id_6': None}]

In [550]:
# koontisivu

line = namedtuple('item', ['selite', 'ignore', 'hinta', 'selite2'])
a = [line(*row) for row in wb['Koontisivu'].iter_rows(values_only=True) if row.count(None) < 3]
b = [x for x in a if isinstance(x.hinta, (int, float))]
c = [x for x in b if (isinstance(x.selite, str) or isinstance(x.selite2, str)) and x.selite != 'Yhteensä']

df = pd.DataFrame(c)
df.iloc[:, [0,2,3]].dropna()


,selite,hinta,selite2
2,0,7058.823529,Siivous
3,0,2.352941,LVI-urakka / sähkö ei tule meiltä
4,0,27058.823529,Lasiseinät
5,ST Team,15294.117647,Lasisiirtoseinät
6,0,1764.705882,Heloitukset
7,0,6117.647059,Ovet
8,0,588.235294,Timanttiporaus


In [581]:
wb.sheetnames

['Tuntihinnoittelut',
 'Materiaalihinnoittelut',
 'Purkutyöt',
 'Väliseinät sisällä',
 'Maalaukset',
 'Alakatot',
 'Laatoitukset',
 'Lattiat',
 'Oviasennukset ja listoitukset',
 'Erillishankinnat',
 'Eritelysivu',
 'Koontisivu',
 'Massalista, rak. tarvikkeet ',
 'Päivitykset pohjaan']

In [619]:
# purkutyöt

line = namedtuple('item', ['a', 'b', 'c'])
lines = [line(*row[:3]) for row in wb['Purkutyöt'].iter_rows(values_only=True) if row.count(None) < 6]

df = pd.DataFrame(lines).iloc[:, 1:].dropna(thresh=2)
df = df[df.iloc[:, 0] != 'Tuntiveloitushinta']

In [662]:
# väliseinät sisällä

line = namedtuple('item', ['a', 'b', 'c'])
lines = [line(*row[:3]) for row in wb['Väliseinät sisällä'].iter_rows(values_only=True)]
df = pd.DataFrame(lines)
df = df[((df.iloc[:, 0].astype(str).str.contains('Hinta', na=False)) |
        (df.iloc[:, 0].astype(str).str.contains('Työ', na=False))) & 
        (pd.to_numeric(df.iloc[:, 2], errors='coerce') != 0)]

In [793]:
# maalaukset

line = namedtuple("Line", ["työ", "hinta"])

lines = [
    line(*row[:2])
    for row in wb["Maalaukset"].iter_rows(values_only=True)
]

pos_blocks = {}
test_blocks = []
current_pos = None

for line in lines:
    # Uusi POS
    if isinstance(line.työ, str) and re.fullmatch(r"POS-\d+", line.työ.strip()):
        current_pos = line.työ.strip()
        pos_blocks[current_pos] = []
        continue

    # ei mitään ennen ekaa POS
    if current_pos is None:
        continue

    # rivien filtteröinti
    if (
        line.työ is not None
        and isinstance(line.hinta, (int, float))
        and line.hinta > 0
    ):
        pos_blocks[current_pos].append(line._asdict())
        test_blocks.append(line._asdict())

df = pd.DataFrame(test_blocks)

In [800]:
# alakatot

line = namedtuple("Line", ["työ", "hinta"])

lines = [
    line(*row[:2])
    for row in wb["Alakatot"].iter_rows(values_only=True)
]

pos_blocks = {}
test_blocks = []
current_pos = None

for line in lines:
    # Uusi POS
    if isinstance(line.työ, str) and re.fullmatch(r"POS-\d+", line.työ.strip()):
        current_pos = line.työ.strip()
        pos_blocks[current_pos] = []
        continue

    # ei mitään ennen ekaa POS
    if current_pos is None:
        continue

    # rivien filtteröinti
    if (
        line.työ is not None
        and isinstance(line.hinta, (int, float))
        and line.hinta > 0
    ):
        pos_blocks[current_pos].append(line._asdict())
        test_blocks.append(line._asdict())

df = pd.DataFrame(test_blocks)
df = df[~df.iloc[:, 0].astype(str).str.contains('Lisä')]

In [801]:
df

,työ,hinta
0,"T-15, T-24, normi",200
1,Tuotteen hinta / m2,25
2,Pientarvikkeet €,300
3,"Hinta, materiaalit",6400
4,"Hinta, Työ",3024
